Loading Packages

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

Loading the Data

In [3]:
df = pd.read_csv(
    "../data/cohort/cohort_analysis.csv",
    parse_dates=["acquisition_date", "cancellation_month"],
)

In [4]:
df['gender'].value_counts(normalize=True)

gender
Female    0.501333
Male      0.498667
Name: proportion, dtype: float64

In [6]:
df['marital_status'].value_counts(normalize=True)

marital_status
Single     0.590583
Married    0.409417
Name: proportion, dtype: float64

In [7]:
df['income_segment'].value_counts(normalize=True)

income_segment
High       0.424224
Medium     0.375319
Low        0.143108
Premium    0.057349
Name: proportion, dtype: float64

In [9]:
df.shape

(12000, 12)

In [10]:
df.info

<bound method DataFrame.info of        user_id acquisition_date cancellation_month  gender marital_status  \
0            1       2024-07-01         2025-02-01    Male        Married   
1            2       2024-04-01         2024-05-01    Male         Single   
2            3       2024-05-01         2024-07-01    Male         Single   
3            4       2024-07-01                NaT    Male        Married   
4            5       2024-03-01         2024-04-01    Male         Single   
...        ...              ...                ...     ...            ...   
11995    11996       2024-02-01                NaT  Female        Married   
11996    11997       2024-06-01         2025-01-01  Female         Single   
11997    11998       2024-01-01         2025-01-01  Female         Single   
11998    11999       2024-01-01                NaT    Male         Single   
11999    12000       2024-03-01         2025-01-01  Female        Married   

       age income_segment         country  

In [11]:
df.isnull().sum()

user_id                  0
acquisition_date         0
cancellation_month    4934
gender                   0
marital_status           0
age                      0
income_segment         631
country                  0
channel                  0
campaign_id              0
device_type              0
plan_type                0
dtype: int64

In [12]:
df.describe(include="all")

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
count,12000.00000,12000,7066,12000,12000,12000.000000,11369,12000,12000,12000,12000,12000
unique,NaN,NaN,NaN,2,2,NaN,4,12,3,9,3,3
top,NaN,NaN,NaN,Female,Single,NaN,High,Netherlands,Paid Ads,Paid Ads_A,Android,Standard
freq,NaN,NaN,NaN,6016,7087,NaN,4823,1057,4854,1636,6582,5979
mean,6000.50000,2024-04-15 20:46:48,2024-07-28 15:31:07.761109,NaN,NaN,34.630250,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00000,2024-01-01 00:00:00,2024-02-01 00:00:00,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,3000.75000,2024-02-01 00:00:00,2024-05-01 00:00:00,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,6000.50000,2024-04-01 00:00:00,2024-07-01 00:00:00,NaN,NaN,34.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,9000.25000,2024-06-01 00:00:00,2024-10-01 00:00:00,NaN,NaN,41.000000,NaN,NaN,NaN,NaN,NaN,NaN
max,12000.00000,2024-08-01 00:00:00,2025-08-01 00:00:00,NaN,NaN,65.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df["acquisition_month"] = df["acquisition_date"].dt.to_period("M")

df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


In [ ]:
df["is_churned"] = df["cancellation_month"].notna().astype(int)
df["is_churned"].value_counts()

is_churned
1    7066
0    4934
Name: count, dtype: int64

In [15]:
df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
df["cancellation_month"] = pd.to_datetime(df["cancellation_month"], errors="coerce")

df["month_churned"] = np.where(
    df["cancellation_month"].notna(),
    (
        (df["cancellation_month"].dt.year - df["acquisition_date"].dt.year) * 12
        + (df["cancellation_month"].dt.month - df["acquisition_date"].dt.month)
    ),
    np.nan
)

df[["user_id", "acquisition_date", "acquisition_month","cancellation_month", "month_churned"]].head(10)

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-02-01,7.0
1,2,2024-04-01,2024-04,2024-05-01,1.0
2,3,2024-05-01,2024-05,2024-07-01,2.0
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0
5,6,2024-08-01,2024-08,NaT,NaN
6,7,2024-05-01,2024-05,NaT,NaN
7,8,2024-05-01,2024-05,2024-06-01,1.0
8,9,2024-07-01,2024-07,NaT,NaN
9,10,2024-02-01,2024-02,2024-12-01,10.0


In [16]:
df["month_churned"] = df["month_churned"].where(
    df["month_churned"].between(1, 12),
    np.nan
)

df["month_churned"].value_counts(dropna=False).sort_index()

month_churned
1.0     2392
2.0     1363
3.0      845
4.0      585
5.0      443
6.0      362
7.0      266
8.0      248
9.0      182
10.0     168
11.0     113
12.0      99
NaN     4934
Name: count, dtype: int64

In [17]:
cohort_size = (
    df.groupby("acquisition_month")
      .agg(new_users=("user_id", "count"))
      .reset_index()
)

cohort_size

,acquisition_month,new_users
0,2024-01,1502
1,2024-02,1528
2,2024-03,1465
3,2024-04,1509
4,2024-05,1541
5,2024-06,1493
6,2024-07,1500
7,2024-08,1462


In [18]:
cohort_churn = (
    df[df["month_churned"].notna()]
    .groupby(["acquisition_month", "month_churned"])
    .agg(users=("user_id", "count"))
    .reset_index()
)

cohort_churn["month_churned"] = cohort_churn["month_churned"].astype(int)

cohort_churn.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


In [19]:
max_month = 12
tenure_range = np.arange(1, max_month + 1)

full_index = pd.MultiIndex.from_product(
    [cohort_size["acquisition_month"].unique(), tenure_range],
    names=["acquisition_month", "month_churned"]
)

cohort_data = (
    cohort_churn
    .set_index(["acquisition_month", "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

cohort_data.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


In [20]:
cohort_data = cohort_data.merge(cohort_size, on="acquisition_month", how="left")
cohort_data.head(15)

,acquisition_month,month_churned,users,new_users
0,2024-01,1,148,1502
1,2024-01,2,95,1502
2,2024-01,3,79,1502
3,2024-01,4,76,1502
4,2024-01,5,52,1502
5,2024-01,6,49,1502
6,2024-01,7,41,1502
7,2024-01,8,37,1502
8,2024-01,9,24,1502
9,2024-01,10,26,1502


In [21]:
cohort_data = cohort_data.sort_values(["acquisition_month", "month_churned"])

cohort_data["cumulative_churn"] = (
    cohort_data.groupby("acquisition_month")["users"].cumsum()
)

cohort_data["active_users"] = (
    cohort_data["new_users"] - cohort_data["cumulative_churn"]
)

cohort_data["retention_rate"] = (
    cohort_data["active_users"] / cohort_data["new_users"]
)

cohort_data["acquisition_month"] = cohort_data["acquisition_month"].astype(str)

cohort_data.head(15)

,acquisition_month,month_churned,users,new_users,cumulative_churn,active_users,retention_rate
0,2024-01,1,148,1502,148,1354,0.901465
1,2024-01,2,95,1502,243,1259,0.838216
2,2024-01,3,79,1502,322,1180,0.785619
3,2024-01,4,76,1502,398,1104,0.735020
4,2024-01,5,52,1502,450,1052,0.700399
5,2024-01,6,49,1502,499,1003,0.667776
6,2024-01,7,41,1502,540,962,0.640479
7,2024-01,8,37,1502,577,925,0.615846
8,2024-01,9,24,1502,601,901,0.599867
9,2024-01,10,26,1502,627,875,0.582557


Retention Line Plots | Gender Level

In [22]:
gender_data = (
    df[df["month_churned"].notna()]
    .groupby(["gender", "month_churned"])
    .agg(churned_users=("user_id", "count"))
    .reset_index()
)
gender_data.head()

,gender,month_churned,churned_users
0,Female,1.0,1200
1,Female,2.0,702
2,Female,3.0,409
3,Female,4.0,294
4,Female,5.0,227


In [23]:
gender_base = (
    df.groupby("gender")
    .agg(total_users=("user_id", "count"))
    .reset_index()
)

In [24]:
gender_data = gender_data.merge(
    gender_base,
    on="gender",
    how="left"
)

gender_data["churn_rate"] = (
    gender_data["churned_users"] / gender_data["total_users"]
)

gender_data.head()

,gender,month_churned,churned_users,total_users,churn_rate
0,Female,1.0,1200,6016,0.199468
1,Female,2.0,702,6016,0.116689
2,Female,3.0,409,6016,0.067985
3,Female,4.0,294,6016,0.048870
4,Female,5.0,227,6016,0.037733


Gender Retention Line Plot

In [25]:
fig = px.line(
    gender_data,
    x="month_churned",
    y="churn_rate",
    color="gender",
    markers=True,
    title="Monthly Churn Rate by Gender",
    labels={
        "month_churned": "Month Since Acquisition",
        "churn_rate": "Churn Rate",
        "gender": "Gender"
    }
)

fig.update_layout(
    yaxis_tickformat=".0%",
    xaxis=dict(dtick=1),
    hovermode="x unified"
)

fig.show()

In [26]:
heatmap_data = cohort_data.pivot(
    index="acquisition_month",
    columns="month_churned",
    values="retention_rate",
)

heatmap_data

month_churned,1,2,3,4,5,6,7,8,9,10,11,12
acquisition_month,,,,,,,,,,,,
2024-01,0.901465,0.838216,0.785619,0.735020,0.700399,0.667776,0.640479,0.615846,0.599867,0.582557,0.575233,0.566578
2024-02,0.898560,0.826571,0.767670,0.726440,0.685864,0.649869,0.628272,0.607330,0.587042,0.569372,0.553010,0.540576
2024-03,0.651195,0.494198,0.423208,0.379522,0.345392,0.325597,0.305802,0.291468,0.283959,0.273038,0.270307,0.264164
2024-04,0.644798,0.487078,0.408880,0.370444,0.341948,0.322068,0.308151,0.298211,0.287608,0.280318,0.273691,0.269052
2024-05,0.755354,0.603504,0.522388,0.467229,0.420506,0.384166,0.363400,0.345230,0.328358,0.315380,0.305646,0.299805
2024-06,0.744139,0.582050,0.473543,0.409243,0.373074,0.344943,0.318151,0.299397,0.287341,0.270596,0.260549,0.252512
2024-07,0.905333,0.823333,0.773333,0.722000,0.688000,0.651333,0.627333,0.596000,0.580000,0.561333,0.550667,0.539333
2024-08,0.903557,0.841313,0.778386,0.733242,0.692886,0.661423,0.638167,0.610807,0.588919,0.578659,0.567031,0.558140


Create the Heatmap

In [27]:
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Blues",
    text_auto=".1%",
    labels={
        "x": "Months Since Acquisition",
        "y": "Acquisition Month",
        "color": "Retention Rate",
    },
    title="Cohort Retention Heatmap"
)

fig.update_layout(
    xaxis_title="Months Since Acquisition",
    yaxis_title="Acquisition Month"
)

fig.show()